In [3]:
# TODO: Importiere torch, numpy, matplotlib
# TODO: Lade Tokenizer und Modell (wie in Notebook 1)
#
# MODEL_NAME = "EleutherAI/pythia-410m"
# n_layers = model.config.num_hidden_layers

from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import torch
import numpy as np
import os
import torch
print(torch.__version__)

import onnxscript
print(onnxscript.__version__)

import onnxruntime
print(onnxruntime.__version__)


MODEL_NAME = "EleutherAI/pythia-410m"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tokenizer laden
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Falls kein pad_token vorhanden
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Modell laden
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",   # wichtig für Attention-Ausgabe
    torch_dtype=torch.float32
)

model.eval()
model.config.use_cache = False

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

prompt = "The movie was good. The sentiment is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

response = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(response)

2.11.0+cpu
0.7.0
1.25.1


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


The movie was good. The sentiment is that your family is in a terrible, terrible situation, and you do everything you can to help them. The characters seem to be very well drawn.

The story is very well done. The story is not so much about the characters as the relationship between those characters and the audience. I really enjoyed the movie.

The characters were very well drawn. The plot was very well constructed. The movie was very good. The sentiment is that your family is in a terrible, terrible situation,


In [4]:

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]
torch.onnx.export(
    model,
    (input_ids, attention_mask),
    "my_model3.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size", 1: "sequence_length"},
    },
    opset_version=17,
    do_constant_folding=True,
    export_params=True,
    verbose=False
)

C:\Users\aroozitalab\AppData\Local\Temp\ipykernel_29352\2829811852.py:3: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0602 14:01:48.548000 29352 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
C:\Users\aroozitalab\AppData\Local\anaconda3\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and tre

ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 17},
            producer_name='pytorch',
            producer_version='2.11.0+cpu',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input_ids"<INT64,[batch_size,sequence_length]>,
                %"attention_mask"<INT64,[batch_size,sequence_length]>
            ),
            outputs=(
                %"logits"<FLOAT,[batch_size,sequence_length,50304]>
            ),
            initializers=(
                %"gpt_neox.embed_in.weight"<FLOAT,[50304,1024]>{TorchTensor(...)},
                %"gpt_neox.layers.0.input_layernorm.weight"<FLOAT,[1024]>{TorchTensor(...)},
                %"gpt_neox.layers.0.input_layernorm.bias"<FLOAT,[1024]>{TorchTensor(...)},
                %"gpt_neox.layers.0.post_attention_layernorm.weight"<FLOAT,[1024]>{TorchTensor(...)},
                %"gpt_neox.layers.0.pos